# Савкина Мария, БД-251м
## Экзаменационное задание

## 1. Подготовка окружения

In [1]:
# Установка необходимых библиотек
!pip install pyspark matplotlib seaborn pandas numpy

In [2]:
# Импорт библиотек
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, avg, stddev, hour, to_timestamp,
    regexp_extract, when, lit, window, desc, asc,
    sum as spark_sum,round as spark_round, rand, randn, least,
    date_format
)

from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType, DoubleType
)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Настройка визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.feature import VectorAssembler, StandardScaler, FeatureHasher

## 2. Инициализация Spark Session

In [3]:
# Создание SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("pw01_var25") \
    .master("local[*]") \
    .config("root.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .config("spark.ui.port", "4040") \
    .config("spark.sql.shuffle.partitions", "300") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "3g") \
    .getOrCreate()

print("SparkSession успешно запущен с расширенными настройками памяти.")

# Установка уровня логирования
spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

26/07/01 00:12:45 WARN Utils: Your hostname, devopsvm resolves to a loopback address: 127.0.1.1; using 192.168.0.137 instead (on interface enp0s3)
26/07/01 00:12:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 00:12:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession успешно запущен с расширенными настройками памяти.
Spark Version: 3.5.3
Spark UI: http://localhost:4040


## 3. Загрузка данных

Выполнить команды из п.3 файла README.md


In [4]:
# Загрузка данных из HDFS в Spark DataFrame
hdfs_path = "hdfs://localhost:9000/user/hadoop/task1/input/brooklyn_sales_map.csv"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(hdfs_path)

print("Схема данных:")
df.printSchema()

print("Общая статистика:")
print(f"Всего записей: {df.count():,}")

print("\nПримеры данных:")
df.show(5, truncate=False)

Схема данных:
root
 |-- _c0: integer (nullable = true)
 |-- borough1: integer (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- building_class_category: string (nullable = true)
 |-- tax_class: string (nullable = true)
 |-- block: integer (nullable = true)
 |-- lot: integer (nullable = true)
 |-- easement: string (nullable = true)
 |-- building_class: string (nullable = true)
 |-- address9: string (nullable = true)
 |-- apartment_number: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- residential_units: integer (nullable = true)
 |-- commercial_units: integer (nullable = true)
 |-- total_units: integer (nullable = true)
 |-- land_sqft: double (nullable = true)
 |-- gross_sqft: double (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- tax_class_at_sale: integer (nullable = true)
 |-- building_class_at_sale: string (nullable = true)
 |-- sale_price: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- year_of_sale: in

Всего записей: 390,883

Примеры данных:


26/07/01 00:12:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+--------+---------------------+-----------------------+---------+-----+----+--------+--------------+-------------------+----------------+--------+-----------------+----------------+-----------+---------+----------+----------+-----------------+----------------------+------------+----------+------------+---------+---+------+------+----------+-------+-------+--------+----------+----------+----------+---------+----------+--------+-------------------+---------+---------+---------+---------+--------+--------+-------+-------+-------+---------+---------+---------+-------+---------+---------+---------------------+-------+--------+-------+-------+----------+----------+----------+---------+----------+---------+----------+--------+---------+--------+----------+--------+--------+---------+---------+---+--------+----------+-------+--------+----------+---------+----------+---------+---------+----------+----------+----------------------------------+--------+--------+--------+-------+--------+----

26/07/01 00:12:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , borough, neighborhood, building_class_category, tax_class, block, lot, easement, building_class, address, apartment_number, zip_code, residential_units, commercial_units, total_units, land_sqft, gross_sqft, year_built, tax_class_at_sale, building_class_at_sale, sale_price, sale_date, year_of_sale, Borough, CD, CT2010, CB2010, SchoolDist, Council, ZipCode, FireComp, PolicePrct, HealthCent, HealthArea, SanitBoro, SanitDistr, SanitSub, Address, ZoneDist1, ZoneDist2, ZoneDist3, ZoneDist4, Overlay1, Overlay2, SPDist1, SPDist2, SPDist3, LtdHeight, SplitZone, BldgClass, LandUse, Easements, OwnerType, OwnerName, LotArea, BldgArea, ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource, NumBldgs, NumFloors, UnitsRes, UnitsTotal, LotFront, LotDepth, BldgFront, BldgDepth, Ext, ProxCode, IrrLotCode, LotType, BsmtCode, AssessLand, AssessTot, ExemptLand, ExemptTo

## 4. Очистка данных (обработка NULL, дубликатов)

In [5]:
# Удаление дубликатов
df = df.dropDuplicates()

# Удаляем строки с критическими пропусками
df = df.dropna(subset=[
    "sale_price",
    "gross_sqft",
    "land_sqft",
    "year_built",
    "tax_class_at_sale"
])

# Остальные заполняем нулями
df = df.fillna({
    "commercial_units":0,
    "residential_units":0,
    "total_units":0
})


# Явное приведение типов

df = df.withColumn("sale_price", col("sale_price").cast(DoubleType())) \
       .withColumn("gross_sqft", col("gross_sqft").cast(DoubleType())) \
       .withColumn("land_sqft", col("land_sqft").cast(DoubleType())) \
       .withColumn("year_built", col("year_built").cast(IntegerType())) \
       .withColumn("total_units", col("total_units").cast(IntegerType()))

print("Количество строк после очистки:", df.count())

26/07/01 00:12:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , borough, neighborhood, building_class_category, tax_class, block, lot, easement, building_class, address, apartment_number, zip_code, residential_units, commercial_units, total_units, land_sqft, gross_sqft, year_built, tax_class_at_sale, building_class_at_sale, sale_price, sale_date, year_of_sale, Borough, CD, CT2010, CB2010, SchoolDist, Council, ZipCode, FireComp, PolicePrct, HealthCent, HealthArea, SanitBoro, SanitDistr, SanitSub, Address, ZoneDist1, ZoneDist2, ZoneDist3, ZoneDist4, Overlay1, Overlay2, SPDist1, SPDist2, SPDist3, LtdHeight, SplitZone, BldgClass, LandUse, Easements, OwnerType, OwnerName, LotArea, BldgArea, ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource, NumBldgs, NumFloors, UnitsRes, UnitsTotal, LotFront, LotDepth, BldgFront, BldgDepth, Ext, ProxCode, IrrLotCode, LotType, BsmtCode, AssessLand, AssessTot, ExemptLand, ExemptTo

Количество строк после очистки: 390883


## 5. Логистическая регрессия (задача классификации)

In [6]:
# Целевой признак - дорогой дом/дешевый дом (выше или ниже медианы)

# Фильтруем явный мусор, цена должна быть рыночной (> $50,000), а площадь здания и год постройки адекватными.
df = df.filter("sale_price > 50000 AND gross_sqft > 100 AND year_built > 1800")

# Считаем честную медиану на очищенных данных
median = df.approxQuantile("sale_price", [0.5], 0.01)[0]
print(f"медиана цены: {median}")

df = df.withColumn(
    "label",
    when(col("sale_price") >= median, 1).otherwise(0)
)

26/07/01 00:13:12 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , borough, neighborhood, building_class_category, tax_class, block, lot, easement, building_class, address, apartment_number, zip_code, residential_units, commercial_units, total_units, land_sqft, gross_sqft, year_built, tax_class_at_sale, building_class_at_sale, sale_price, sale_date, year_of_sale, Borough, CD, CT2010, CB2010, SchoolDist, Council, ZipCode, FireComp, PolicePrct, HealthCent, HealthArea, SanitBoro, SanitDistr, SanitSub, Address, ZoneDist1, ZoneDist2, ZoneDist3, ZoneDist4, Overlay1, Overlay2, SPDist1, SPDist2, SPDist3, LtdHeight, SplitZone, BldgClass, LandUse, Easements, OwnerType, OwnerName, LotArea, BldgArea, ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource, NumBldgs, NumFloors, UnitsRes, UnitsTotal, LotFront, LotDepth, BldgFront, BldgDepth, Ext, ProxCode, IrrLotCode, LotType, BsmtCode, AssessLand, AssessTot, ExemptLand, ExemptTo

медиана цены: 560000.0


In [7]:
#Feature Engineering
#возраст здания на момент продажи
df = df.withColumn("building_age", col("year_of_sale") - col("year_built"))

# была ли реконструкция здания
df = df.withColumn("is_altered", when(col("YearAlter1").cast("int") > 0, 1).otherwise(0))

# Признак коммерческой недвижимости
df = df.withColumn("is_commercial", when(col("commercial_units") > 0, 1).otherwise(0))

# Заполняем возможные null в новых колонках
df = df.fillna({"building_age": 0, "is_altered": 0, "is_commercial": 0})

In [8]:
# Перечисляем числовые колонки для использования
numeric_cols = ["gross_sqft", "land_sqft", "year_built", "residential_units", "commercial_units", "total_units"]

# Переводим их в double и заполняем null нулями , чтобы не было ошибок
for c in numeric_cols:
    df = df.withColumn(c, col(c).cast("double")).fillna(0, subset=[c])

In [9]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Индексация района
neighborhood_indexer = StringIndexer(
    inputCol="neighborhood",
    outputCol="neighborhood_indexed",
    handleInvalid="keep"
)

# Индексация категории здания
build_cat_indexer = StringIndexer(
    inputCol="build_cat",
    outputCol="build_cat_indexed",
    handleInvalid="keep"
)

hasher = FeatureHasher(
    numFeatures=32, 
    inputCols=["neighborhood", "building_class_category"], 
    outputCol="categorical_features"
)

In [10]:
assembler = VectorAssembler(
    inputCols=[
        "gross_sqft", 
        "land_sqft", 
        "building_age",
        "is_altered",
        "is_commercial",
        "neighborhood_Vec",
        "build_cat_Vec"
    ],
    outputCol="unscaled_features" 
)

In [11]:
# Масштабируем признаки, чтобы привести площади, возраст и векторы к единому масштабу
scaler = StandardScaler(
    inputCol="unscaled_features", 
    outputCol="features", 
    withStd=True, 
    withMean=False
)

In [12]:
# Разделяем выборку
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# VectorAssembler
assembler = VectorAssembler(
    inputCols=[
        "categorical_features", 
        "gross_sqft", 
        "land_sqft", 
        "year_built", 
        "total_units", 
        "sale_price"
    ], 
    outputCol="features_unscaled",
    handleInvalid="skip"
)

# Настраиваем Scaler (обязателен для логистической регрессии)
scaler = StandardScaler(
    inputCol="features_unscaled", 
    outputCol="features", 
    withStd=True, 
    withMean=False
)

# Логистическая регрессия с сильной регуляризацией
lr = LogisticRegression(
    featuresCol="features", 
    labelCol="label", 
    maxIter=15, 
    regParam=0.1
)

pipeline = Pipeline(stages=[
    hasher,    
    assembler,
    scaler,    
    lr         
])

# Обучаем модель
pipeline_model = pipeline.fit(train_data)

# Делаем предсказание
predictions = pipeline_model.transform(test_data)

# Оцениваем точность
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)

print(f"\nТочность логистической регрессии (Accuracy): {accuracy * 100:.2f}%")

26/07/01 00:13:19 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , borough, neighborhood, building_class_category, tax_class, block, lot, easement, building_class, address, apartment_number, zip_code, residential_units, commercial_units, total_units, land_sqft, gross_sqft, year_built, tax_class_at_sale, building_class_at_sale, sale_price, sale_date, year_of_sale, Borough, CD, CT2010, CB2010, SchoolDist, Council, ZipCode, FireComp, PolicePrct, HealthCent, HealthArea, SanitBoro, SanitDistr, SanitSub, Address, ZoneDist1, ZoneDist2, ZoneDist3, ZoneDist4, Overlay1, Overlay2, SPDist1, SPDist2, SPDist3, LtdHeight, SplitZone, BldgClass, LandUse, Easements, OwnerType, OwnerName, LotArea, BldgArea, ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource, NumBldgs, NumFloors, UnitsRes, UnitsTotal, LotFront, LotDepth, BldgFront, BldgDepth, Ext, ProxCode, IrrLotCode, LotType, BsmtCode, AssessLand, AssessTot, ExemptLand, ExemptTo


Точность логистической регрессии (Accuracy): 69.88%


In [13]:
#DataLens
import os


datalens_df = predictions.select(
    "neighborhood",
    "building_class_category",
    "gross_sqft",
    "sale_price",
    "tax_class_at_sale", 
    "label",              
    "prediction"          
)

local_pdf = datalens_df.toPandas()

output_path = "model_results_for_datalens.csv"

local_pdf.to_csv(output_path, index=False, encoding='utf-8')

print(f"Файл успешно сохранен локально.")
print(f"Имя файла для скачивания: {os.path.abspath(output_path)}")
print(f"Всего строк для выгрузки в DataLens: {len(local_pdf)}")

26/07/01 00:13:57 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , borough, neighborhood, building_class_category, tax_class, block, lot, easement, building_class, address, apartment_number, zip_code, residential_units, commercial_units, total_units, land_sqft, gross_sqft, year_built, tax_class_at_sale, building_class_at_sale, sale_price, sale_date, year_of_sale, Borough, CD, CT2010, CB2010, SchoolDist, Council, ZipCode, FireComp, PolicePrct, HealthCent, HealthArea, SanitBoro, SanitDistr, SanitSub, Address, ZoneDist1, ZoneDist2, ZoneDist3, ZoneDist4, Overlay1, Overlay2, SPDist1, SPDist2, SPDist3, LtdHeight, SplitZone, BldgClass, LandUse, Easements, OwnerType, OwnerName, LotArea, BldgArea, ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource, NumBldgs, NumFloors, UnitsRes, UnitsTotal, LotFront, LotDepth, BldgFront, BldgDepth, Ext, ProxCode, IrrLotCode, LotType, BsmtCode, AssessLand, AssessTot, ExemptLand, ExemptTo

Файл успешно сохранен локально.
Имя файла для скачивания: /home/devops/Downloads/model_results_for_datalens.csv
Всего строк для выгрузки в DataLens: 28863


#Остановка Spark

In [14]:
# Остановка SparkSession
spark.stop()
print("SparkSession остановлен")

SparkSession остановлен
